# 04 — Cross-Layer Temporal Alignment

For each target time step (daily), define the best-available-scene logic per source given gaps.

This notebook is likely the most methodologically consequential step in the EDA. The gap-fill strategy established here becomes the documented assumption underpinning the entire ET model.

**Key questions to resolve:**
1. What is the actual cloud-masked pixel fraction per MODIS LST scene over the AOI?
2. What is the maximum contiguous gap length for LST during wet-season months?
3. For each source, what is the gap-fill rule (nearest valid, N-day window, monthly climatology)?
4. What is the maximum gap before a time step is excluded from the dataset entirely?

**Inputs:** `data/processed/aligned/` (from `03_crs_resolution.ipynb`)  
**Gates:** `05_distributions_correlations.ipynb`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rioxarray
import xarray as xr

ALIGNED_DIR = Path("../../data/processed/aligned")
FIG_DIR     = Path("../../outputs/figures/eda")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Build a data availability calendar

For each day in the sample period, record:
- Whether a valid (non-masked) LST scene exists
- Valid pixel fraction across the AOI
- Whether a NDVI composite is current (16-day window)
- Whether SMAP retrieval exists (microwave — should be near-complete)
- Whether GPM / ERA5 are available (should be gap-free)

In [ ]:
# TODO: iterate over aligned LST files, compute valid pixel fraction per scene
# availability = []
# for f in sorted(ALIGNED_DIR.glob("lst_aligned_*.tif")):
#     da = rioxarray.open_rasterio(f).squeeze()
#     valid_frac = float((da > 0).sum() / da.size)
#     date = pd.Timestamp(f.stem.split("_")[-1])  # adjust parse logic to filename pattern
#     availability.append({"date": date, "lst_valid_frac": valid_frac})
# avail_df = pd.DataFrame(availability).set_index("date")
# print(avail_df.describe())
print("Availability calendar: stub")

## 2. Visualise the gap calendar

In [ ]:
# TODO: calendar heatmap of valid pixel fraction
# fig, ax = plt.subplots(figsize=(14, 3))
# ax.bar(avail_df.index, avail_df["lst_valid_frac"], color="steelblue", width=1)
# ax.axhline(0.7, color="red", linestyle="--", label="70% threshold")
# ax.set_ylabel("Valid pixel fraction")
# ax.set_title("MODIS LST data availability — Greater Accra 2023")
# ax.legend()
# fig.savefig(FIG_DIR / "lst_availability_calendar.png", dpi=150, bbox_inches="tight")
print("Gap calendar plot: stub")

## 3. Gap statistics

In [ ]:
# TODO: compute gap run lengths
# THRESHOLD = 0.70  # scenes below this fraction treated as unusable
# is_gap = avail_df["lst_valid_frac"] < THRESHOLD
#
# # Run-length encode gaps
# from itertools import groupby
# runs = [(k, sum(1 for _ in g)) for k, g in groupby(is_gap)]
# gap_lengths = [n for is_gap_run, n in runs if is_gap_run]
#
# print(f"Total gap days: {sum(gap_lengths)}")
# print(f"Max contiguous gap: {max(gap_lengths)} days")
# print(f"Gap length distribution: {sorted(gap_lengths, reverse=True)[:10]}")
print("Gap statistics: stub")

## 4. Define gap-fill rules

Based on gap statistics above, propose and test fill rules. Document the final choice here — it will be reproduced verbatim in `EDA_FINDINGS.md`.

**Candidate rules for LST (most critical source):**

| Rule | Description | Risk |
|------|-------------|------|
| Nearest valid scene ≤ N days | Use closest non-gap scene | N too large → stale data in fast-changing conditions |
| Linear interpolation between bracketing scenes | Smooth fill for short gaps | Can introduce spurious trends |
| Monthly climatology | Long-term mean for that DOY | Conservative; loses inter-annual signal |
| Exclude time step | Don't fill; drop from dataset | Safest; reduces temporal coverage |

In [ ]:
# TODO: implement and compare rules on a known gap period
# Show filled vs. true values where both are available (hold-out test)
print("Gap-fill rule comparison: stub")

## 5. Apply chosen gap-fill and assemble aligned time series

Output: one xarray Dataset per time step with all sources on the common 1km grid.

In [ ]:
# TODO: after gap-fill rule is decided, loop over target dates and build the aligned dataset
# Flag any time step where gap-fill was applied (add a QA flag variable to the dataset)
print("Aligned dataset assembly: stub")

## 6. Temporal alignment decisions — final record

Fill in after completing sections 3–5. Copy to `EDA_FINDINGS.md` section 4.

- **Threshold for scene exclusion:** <!-- e.g. <70% valid pixels → treat as gap -->
- **LST gap-fill rule:** <!-- e.g. nearest valid within 5 days, else exclude -->
- **NDVI gap-fill rule:** <!-- 16-day composite should be near gap-free; note any exceptions -->
- **Max gap found (wet season):** <!-- days -->
- **QA flag convention:** <!-- 0 = direct observation, 1 = gap-filled, 2 = excluded -->